# LeapSinger — Re-synthesis (v/uv-less vs v/uv)

Turn a phoneme sequence + **durations** + an **F0 contour** into a singing waveform:

```
phonemes, durations, F0  ──►  acoustic model (harmonic flow)  ──►  mel  ──►  NHVSing vocoder (ONNX)  ──►  wav
```

The duration and F0 models are **separate projects** — this notebook consumes their output.
The NHVSing vocoder is bundled as an ONNX (`checkpoints/nhv_v3.onnx`).

LeapSinger ships **two acoustic models** that differ in one design choice — whether the model is
told where the voice is **voiced vs unvoiced (v/uv)**:

- **v/uv-less** (`3speaker_gan2d`) — the model sees only a gap-less (interpolated) F0. Its
  excitation (the flow's starting point) carries harmonics across **every** frame.
- **v/uv** (`3singer_ritsu3style_uv_gan2d`) — the model also gets a voiced/unvoiced flag, and its
  excitation **switches the harmonics off in unvoiced frames**.

We resynthesize the same phrase with both, and plot how their **pseudo-mel** (the excitation) differs.

> Run this from the `notebooks/` folder. Put the model checkpoints in `sample_data/` first
> (see `sample_data/place_model_here.txt`).

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, display

REPO = ".."                                          # repo root (this notebook lives in notebooks/)
sys.path.insert(0, REPO)
from infer import load_acoustic, infer_mel, load_vocoder, mel_to_wav

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
VOCODER = "../checkpoints/nhv_v3.onnx"               # bundled NHVSing vocoder ONNX (mel -> wav)

# Two acoustic models: without and with the voiced/unvoiced (v/uv) condition.
model_nouv, cfg_nouv = load_acoustic("sample_data/3speaker_gan2d.pth", device=DEVICE)
model_uv,   cfg_uv   = load_acoustic("sample_data/3singer_ritsu3style_uv_gan2d.pth", device=DEVICE)
vocoder = load_vocoder(VOCODER)
SR = vocoder.sr
STEPS = int(cfg_nouv.get("infer_steps", 1))          # reflow: one step is the deployed setting

for tag, m, c in [("v/uv-less", model_nouv, cfg_nouv), ("v/uv", model_uv, cfg_uv)]:
    print(f"{tag:9s}: use_uv={m.use_uv}  n_styles={c.get('n_styles')}  infer_steps={c.get('infer_steps')}")

def show(wav, mel=None, title=""):
    if mel is not None:
        plt.figure(figsize=(9, 2.2))
        plt.imshow(mel, origin="lower", aspect="auto", cmap="magma", vmin=-11.5, vmax=2.0)
        plt.title(title, loc="left", fontsize=9); plt.axis("off"); plt.show()
    display(Audio(wav, rate=SR))

## 1. Input — a phrase from `sample_data/`

Self-contained: take the ground-truth phrase (`ritsu_flashblack_0000`), read the phoneme timing from
its `.lab`, and extract the F0 and the voiced/unvoiced flag with RMVPE. In practice these come from
your duration and F0 models. We cut at a pause near 6 s to keep the clip short.

In [ ]:
import librosa
from leapsinger.config import MelSpec
from preprocess.lab import load_lab
from preprocess.f0_rmvpe import extract_f0_rmvpe
from preprocess.vocab import PHONEME2ID

mel_cfg = MelSpec(hop=256)
SR_, HOP, FPS = mel_cfg.sr, mel_cfg.hop, mel_cfg.frame_rate

TARGET_SECONDS = 6.0
lab_rows = load_lab("sample_data/ritsu_flashblack_0000.lab", lab_unit="sec")
pau_ends = [e for s, e, ph in lab_rows if ph == "pau" and e > 1.0]
cut_sec  = min(pau_ends, key=lambda t: abs(t - TARGET_SECONDS)) if pau_ends else None

wav_gt, _ = librosa.load("sample_data/ritsu_flashblack_0000.wav", sr=SR_, mono=True, duration=cut_sec)
n_frames = len(wav_gt) // HOP
rows = [(s, min(e, n_frames / FPS), ph) for s, e, ph in lab_rows if s < n_frames / FPS]

# F0 + voiced/unvoiced flag (gap-less F0 via interpolation; the flag marks the real voiced frames)
f0, voiced = extract_f0_rmvpe(np.clip(wav_gt, -1.0, 1.0), SR_, HOP,
                              fmin=150.0, fmax=1000.0, device=DEVICE, interpolate=True)
f0, voiced = f0[:n_frames], voiced[:n_frames]
logf0 = np.log2(np.maximum(f0, 1.0)).astype(np.float32)

# phonemes -> ids, and per-phoneme frame counts from the lab (sum stays exact)
phonemes = [ph for _, _, ph in rows]
ph_ids   = np.array([PHONEME2ID[ph] for ph in phonemes], np.int64)
ends = np.cumsum([e - s for s, e, _ in rows])
fb = np.clip(np.round(np.concatenate([[0.0], ends]) * FPS).astype(int), 0, n_frames); fb[-1] = n_frames
ph_durs = np.maximum(np.diff(fb), 1); ph_durs[np.argmax(ph_durs)] += n_frames - ph_durs.sum()

print(f"{len(wav_gt)/SR_:.1f} s, {n_frames} frames, {len(phonemes)} phonemes, voiced={voiced.mean()*100:.0f}%")
print("first phonemes:", " ".join(phonemes[:16]))

## 2. Re-synthesize with each model

Same phonemes, durations and F0 for both models. Namine Ritsu is speaker `2` in both.

In [ ]:
def resynth(model):
    item = dict(ph_ids=ph_ids, ph_durs=ph_durs, f0_logf0=logf0,
                uv=voiced.astype(np.float32), spk_id=2, style_id=0)
    mel = infer_mel(model, item, num_steps=STEPS, device=DEVICE)
    wav = mel_to_wav(vocoder, mel, logf0, voiced)
    return mel, wav

for tag, m in [("v/uv-less  (3speaker_gan2d)", model_nouv),
               ("v/uv       (3singer_ritsu3style_uv)", model_uv)]:
    mel, wav = resynth(m)
    print(tag)
    show(wav, mel, f"resynth · {tag}")

## 3. The pseudo-mel — the flow's starting point

LeapSinger's flow does not start from noise; it starts from a **pseudo-mel**: an F0-harmonic
excitation (impulse harmonics + a little white noise). The flow only has to shape the formant
envelope on top of it.

This is exactly where the two models differ. Below we plot each model's excitation for the same
input. The **v/uv-less** model lays harmonics across every frame (it never sees where the voice
stops); the **v/uv** model **switches the harmonics off in unvoiced frames**, leaving only noise — so
consonants and gaps start from silence rather than from a buzz. The dark vertical bands in the
bottom panel line up with the `0`s of the voiced flag on top.

In [ ]:
def excitation_mel(model, cfg):
    f0l = torch.as_tensor(logf0, dtype=torch.float32)[None].to(DEVICE)
    uvt = torch.as_tensor(voiced.astype(np.float32), dtype=torch.float32)[None].to(DEVICE)
    torch.manual_seed(0)                             # fix the noise so the two panels are comparable
    with torch.no_grad():
        x0 = model._excitation_x0(f0l, uvt)          # [1, mel, T] in the flow's normalized domain
    vmin, vmax = cfg["mel_vmin"], cfg["mel_vmax"]
    return (x0[0].cpu().numpy() + 1) / 2 * (vmax - vmin) + vmin     # -> mel domain

exc_nouv = excitation_mel(model_nouv, cfg_nouv)
exc_uv   = excitation_mel(model_uv,   cfg_uv)

fig, ax = plt.subplots(3, 1, figsize=(10, 5.2), sharex=True)
ax[0].plot(voiced.astype(float)); ax[0].set_ylim(-0.1, 1.1); ax[0].set_yticks([0, 1])
ax[0].set_title("voiced (v/uv) flag  — 1 = voiced", loc="left", fontsize=9)
for a, m, t in [(ax[1], exc_nouv, "pseudo-mel — v/uv-less (harmonics in every frame)"),
                (ax[2], exc_uv,   "pseudo-mel — v/uv (harmonics gated to voiced frames)")]:
    a.imshow(m, origin="lower", aspect="auto", cmap="magma", vmin=-11.5, vmax=2.0)
    a.set_title(t, loc="left", fontsize=9); a.set_ylabel("mel", fontsize=7)
ax[2].set_xlabel("frame", fontsize=8)
plt.tight_layout(); plt.show()